# Neo4j — connect and explore

Uses only **`NEO4J_URI`**, **`NEO4J_USERNAME`**, **`NEO4J_PASSWORD`** from your project `.env` (same as `store.py`).

Open this notebook with the **project root** (`graph_rag`) as the working directory, then pick a kernel that has `neo4j` + `python-dotenv` (e.g. `uv run python` / your venv).

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
# Optional: set NEO4J_DATABASE in .env for multi-DB (Aura / Enterprise); otherwise default DB is used
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE") or None

In [2]:
def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))


def run_cypher(cypher: str, params: dict | None = None) -> list[dict]:
    params = params or {}
    driver = get_driver()
    try:
        session_context = (
            driver.session(database=NEO4J_DATABASE)
            if NEO4J_DATABASE
            else driver.session()
        )
        with session_context as session:
            result = session.run(cypher, **params)
            return [r.data() for r in result]
    finally:
        driver.close()


# smoke test: connection + server responds
rows = run_cypher("RETURN 1 AS ok")
print("Connected.", rows)

Connected. [{'ok': 1}]


## What’s in the graph?
Labels, relationship types, and rough node counts (handy after `store.py` + `load_13f_graph.py`).

In [3]:
labels = run_cypher("CALL db.labels() YIELD label RETURN label ORDER BY label")
display(pd.DataFrame(labels))

rel_types = run_cypher("CALL db.relationshipTypes() YIELD relationshipType AS type RETURN type ORDER BY type")
display(pd.DataFrame(rel_types))

,label
0,Chunk
1,Company
2,Filing
3,Manager
4,Section


,type
0,CONTAINS
1,FILED
2,HAS_FIRST_CHUNK
3,NEXT
4,OWNS_STOCK_IN
5,PART_OF


In [4]:
counts = run_cypher(
    """
    MATCH (n)
    UNWIND labels(n) AS label
    RETURN label, count(*) AS nodes
    ORDER BY nodes DESC
    """
)
display(pd.DataFrame(counts))

,label,nodes
0,Manager,1065
1,Chunk,593
2,Section,33
3,Filing,2
4,Company,1


## Peek at your project’s data (edit freely)
Uncomment or change limits / filters.

In [5]:
# A few Filings (from store.py)
display(pd.DataFrame(run_cypher(
    "MATCH (f:Filing) RETURN f.filing_type AS filing_type, f.period AS period, f.filename AS filename LIMIT 20"
)))

# 13F: MSCI + sample managers (from load_13f_graph.py)
display(pd.DataFrame(run_cypher(
    """
    MATCH (c:Company {ticker: 'MSCI'})<-[r:OWNS_STOCK_IN]-(m:Manager)
    RETURN m.name AS manager, m.address AS address, r.value AS value, r.shares AS shares, r.as_of AS as_of
    ORDER BY r.value DESC
    LIMIT 10
    """
)))

# Chunks: one line preview (vector index is separate; this is plain graph)
display(pd.DataFrame(run_cypher(
    """
    MATCH (c:Chunk)
    RETURN c.filing_type AS filing, c.period AS period, c.section AS section, c.page_number AS page, c.chunk_index AS idx,
           left(c.page_content, 200) AS preview
    LIMIT 5
    """
)))

,filing_type,period,filename
0,10-K,FY2025,msci_10k_fy2025.htm
1,10-Q,Q1 2026,msci_10q_q1_2026.htm


,manager,address,value,shares,as_of
0,VANGUARD GROUP INC,"Po Box 2600, V26, Valley Forge, PA, 19482-2600",5.373393e+09,9365717.0,31-DEC-2025
1,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001",3.385625e+09,5901077.0,31-DEC-2025
2,STATE STREET CORP,"1 CONGRESS STREET, SUITE 1, BOSTON, MA, 02114",1.828324e+09,3186733.0,31-DEC-2025
3,BAMCO INC /NY/,"767 FIFTH AVENUE, 49TH FL, NEW YORK, NY, 10153",1.581055e+09,2755747.0,31-DEC-2025
4,MORGAN STANLEY,"1585 BROADWAY, NEW YORK, NY, 10036",1.415713e+09,2467556.0,31-DEC-2025
5,"GEODE CAPITAL MANAGEMENT, LLC","100 SUMMER STREET, 12TH FLOOR, BOSTON, MA, 02110",1.141003e+09,1993077.0,31-DEC-2025
6,PRINCIPAL FINANCIAL GROUP INC,"711 HIGH STREET, DES MOINES, IA, 50392",9.853374e+08,1717364.0,31-DEC-2025
7,EDGEWOOD MANAGEMENT LLC,"600 STEAMBOAT ROAD, SUITE 103, GREENWICH, CT, ...",8.072714e+08,1407058.0,31-DEC-2025
8,POLEN CAPITAL MANAGEMENT LLC,"1825 Nw Corporate Boulevard, Suite 300, Boca R...",7.636836e+08,1331085.0,31-DEC-2025
9,PineStone Asset Management Inc.,"1981 McGill College Avenue, Suite 1600, Montre...",5.742705e+08,1000942.0,31-DEC-2025


,filing,period,section,page,idx,preview
0,10-K,FY2025,Preamble,1,0,msci-20251231\nTable of Contents\nUNITED STATE...
1,10-K,FY2025,Preamble,1,1,[TABLE]\nDelaware | 13-4038723\n(State or Othe...
2,10-K,FY2025,Preamble,1,2,"7 World Trade Center\n250 Greenwich Street, 49..."
3,10-K,FY2025,Preamble,1,3,[TABLE]\nTitle of each class | Trading Symbol(...
4,10-K,FY2025,Preamble,1,4,Securities registered pursuant to Section 12(g...


In [6]:
display(pd.DataFrame(run_cypher(
    """
    CALL db.index.fulltext.queryNodes("managerNameFulltext", "BlackRock")
    YIELD node, score
    RETURN node.name AS name, node.cik AS cik, node.address AS address, score
    ORDER BY score DESC
    LIMIT 20
    """
)))

,name,cik,address,score
0,"BlackRock, Inc.",0002012383,"50 Hudson Yards, New York, NY, 10001",3.739539


In [12]:
display(pd.DataFrame(run_cypher(
    """ 
  MATCH p=(:Manager)-[:LOCATED_AT]->(address:Address)
  RETURN address.state as state, count(address.state) as numManagers
  ORDER BY numManagers DESC
  LIMIT 10
    """
)))


,state,numManagers
0,NY,109
1,CA,89
2,TX,55
3,MA,46
4,FL,44
5,OH,37
6,PA,37
7,IL,34
8,CT,30
9,NJ,29


In [14]:
display(pd.DataFrame(run_cypher(
    """ 
MATCH (m:Manager)-[:LOCATED_AT]->(a:Address)
WHERE a.state = 'NY' AND a.raw CONTAINS ', NY,'
WITH m, trim(last(split(split(a.raw, ', NY,')[0], ','))) AS place
RETURN place, count(DISTINCT m) AS num_investment_firms
ORDER BY num_investment_firms DESC
LIMIT 20
    """
)))


,place,num_investment_firms
0,NEW YORK,73
1,New York,20
2,ALBANY,2
3,BUFFALO,1
4,Fairport,1
5,WILLIAMSVILLE,1
6,Cobleskill,1
7,ITHACA,1
8,CANANDAIGUA,1
9,NEW HARTFORD,1


In [15]:
display(pd.DataFrame(run_cypher(
    """ 
CALL db.index.fulltext.queryNodes("managerNameFulltext", "blackrock*")
YIELD node, score
WITH node AS b, score ORDER BY score DESC LIMIT 1
MATCH (b)-[:LOCATED_AT]->(ba:Address)
WHERE ba.state = 'NY' AND ba.raw CONTAINS ', NY,'
WITH b, ba, trim(last(split(split(ba.raw, ', NY,')[0], ','))) AS b_place
MATCH (mgr:Manager)-[:LOCATED_AT]->(a:Address)
WHERE b <> mgr AND a.state = 'NY' AND a.raw CONTAINS ', NY,'
  AND trim(last(split(split(a.raw, ', NY,')[0], ','))) = b_place
RETURN b.name, ba.raw, mgr.name, a.raw
LIMIT 25
    """
)))


,b.name,ba.raw,mgr.name,a.raw
0,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001",TOCQUEVILLE ASSET MANAGEMENT L.P.,"40 West 57th Street, 19th Floor, New York, NY,..."
1,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001",MUTUAL OF AMERICA CAPITAL MANAGEMENT LLC,"320 Park Avenue, New York, NY, 10022"
2,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001","D. E. Shaw & Co., Inc.","Two Manhattan West, 375 Ninth Avenue, 52nd Flo..."
3,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001",NEW YORK LIFE INVESTMENT MANAGEMENT LLC,"51 MADISON AVE, New York, NY, 10010"
4,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001","Rafferty Asset Management, LLC","535 Madison Ave,, 37th Floor, New York, NY, 10022"
5,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001","Evercore Wealth Management, LLC","55 East 52nd Street, 23rd Floor, New York, NY,..."
6,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001",Virtu Financial LLC,"1633 Broadway, New York, NY, 10019"
7,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001",Kemnay Advisory Services Inc.,"1 Rockefeller Plaza, Suite 1800, New York, NY,..."
8,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001",Tsai Capital Corp,"20 East End Avenue, New York, NY, 10028"
9,"BlackRock, Inc.","50 Hudson Yards, New York, NY, 10001","PATTON FUND MANAGEMENT, INC.","228 Park Avenue S, Suite 31616, New York, NY, ..."
